# Validate the federated model

Last step of the training lifecycle: the federated run is finished and the global LoRA adapter was copied
from the hub (`make fetch-model`). Here we check it against two references:

| Model | What it is |
|---|---|
| **base** | Granite 4.0 350M with no fine-tuning |
| **brasil-only** | Same LoRA recipe, trained only on the Brazil branch's data, with the same compute Brazil spent in the federation (`make baseline`) |
| **federated** | The adapter produced by the three branches through Flower + OCM, without any message leaving its branch |

The test set (`data/test_balanced.jsonl`) has every queue equally represented, which no single branch has.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
sys.path.insert(0, str(ROOT / "app")); sys.path.insert(0, str(ROOT / "eval"))

import evaluate as ev
from bank_router import task
from IPython.display import Markdown, display

device = task.pick_device()
models = ev.available_models()
print("device:", device, "| models found:", models)

## The data each branch trained on
Non-IID on purpose: each branch is dominated by its most frequent subject.

In [ ]:
stats = json.loads((ROOT / "data" / "stats.json").read_text())
rows = ["| Branch | " + " | ".join(task.QUEUES) + " | total |", "|---|" + "---|" * (len(task.QUEUES) + 1)]
for branch, s in stats["clients"].items():
    counts = s["train_by_queue"]
    rows.append(f"| {branch} | " + " | ".join(str(counts.get(q, 0)) for q in task.QUEUES) + f" | {s['train']} |")
display(Markdown("\n".join(rows)))

## How the federated run progressed
Accuracy of the global model on each branch's own validation set, averaged by the hub after every round.

In [ ]:
import matplotlib.pyplot as plt
hist_file = ROOT / "models" / "adapters" / "federated" / "history.json"
if hist_file.exists():
    hist = json.loads(hist_file.read_text())
    rounds = sorted(int(r) for r in hist["evaluate"])
    acc = [hist["evaluate"][str(r)]["accuracy"] for r in rounds]
    loss = [hist["train"][str(r)]["train_loss"] for r in rounds]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(rounds, acc, marker="o"); ax[0].set(title="Validation accuracy (branches)", xlabel="round", ylim=(0, 1))
    ax[1].plot(rounds, loss, marker="o", color="tab:red"); ax[1].set(title="Training loss", xlabel="round")
    plt.tight_layout(); plt.show()
    print(f"run {hist['run_id']}: {len(rounds)} rounds in {hist['seconds']:.0f}s")
else:
    print("No federated adapter yet: run the federation and `make fetch-model`.")

## Accuracy per queue

In [ ]:
test = task.load_records(str(ROOT / "data" / "test_balanced.jsonl"))
results = ev.evaluate_all(models, test, device)
display(Markdown(ev.summary_markdown(results, len(test))))

## Try your own message

In [ ]:
from functools import lru_cache
tok = task.load_tokenizer()

@lru_cache(maxsize=None)
def get(name):
    return ev.load(name, device)

def classify(message: str):
    rows = ["| Model | Queue | Confidence |", "|---|---|---|"]
    for name in models:
        p = task.predict(get(name), tok, [message])[0]
        rows.append(f"| {name} | {p['queue']} | {p['probs'][p['queue']]:.0%} |")
    display(Markdown(f'**"{message}"**\n\n' + "\n".join(rows)))

classify("I was charged twice for the same coffee in Lisbon.")